RAG Using text-embedding-3-small model 

In [14]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

faiss_index = FAISS.load_local(
    "faiss_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [15]:
import pandas as pd
df=pd.read_excel("Question and answer.xlsx")

In [16]:
df["Question "]

0     What does Article 150 of the Constitution of I...
1         Where does the payment process in PFMS start?
2     What must be done when discrepancies are notic...
3     What should be done when cheques remain un-enc...
4     Who prepares a consolidated monthly account fo...
5     Who maintains the G.P.F. accounts of All India...
6     What is the mandatory contribution rate under ...
7     Who is responsible for maintaining the detaile...
8     What does the Finance Accounts of the Central ...
9     What does Statement No. 6 in Part II of the Fi...
10    Why is reconciliation necessary between the ac...
11    When did the scheme of “One Bank–One Commissio...
12    According to Rule 96 of the CGST Rules, when i...
13    When was the Online Tax Accounting System (OLT...
14    How often should important offices undergo Int...
15    Who is responsible for ensuring accurate data ...
16    Who prepares the "Due and Drawn Statement" for...
17    What action should the Pay and Accounts Of

In [17]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")


In [18]:
con=[]
ans=[]
for i,query in enumerate(list(df["Question "])):
    print(f"question {i} = {query}")
    results1 = faiss_index.similarity_search(query, k=1)
    context=results1[0].page_content
    con.append(context)
    print(f"context {i} = {context}")
    prompt = f"""
    Use the context to answer the question, answer lies only in the context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    answer=llm.invoke(prompt).content
    print(f"answer {i} = {answer}")
    ans.append(answer)

question 0 = What does Article 150 of the Constitution of India provide regarding Government Accounts?
context 0 = 1.1.1   Article 150 of the Constitution of India provides for the maintenance of Government 
Accounts in such form as the President may, on the advice of the Comptroller and Auditor -
General of India, prescribe.   In exercise of these powers , basic rules relating to the Form of 
Accounts were  framed  in the form of ‘ Government Accounting Rules ’ (GAR) . The Civil Accounts 
Manual is intended to guide the Civil Ministries/ Departments of Central Government in carrying
answer 0 = Article 150 of the Constitution of India provides for the maintenance of Government Accounts in such form as prescribed by the President on the advice of the Comptroller and Auditor-General of India.
question 1 = Where does the payment process in PFMS start?
context 1 = through the PFMS module. Process flow for  the same is depicted below: - 
 
 366  
  
1. Payment Gateway receives the fund from

In [19]:
df["context"]=con

In [20]:
df["answer"]=ans

In [21]:
df.to_excel("QandA_embedding_3_small.xlsx")

RAG using Keyword search

In [ ]:
all_docs = list(faiss_index.docstore._dict.values())

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(page_content=doc.page_content, metadata={"source": "pdf", "chunk_id": doc.metadata["chunk_id"]})
    for doc in all_docs
]

In [22]:
import pandas as pd
df=pd.read_excel("Question and answer.xlsx")

In [ ]:
from langchain_community.retrievers import BM25Retriever
retriever = BM25Retriever.from_documents(documents,k=1)

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
con=[]
ans=[]
for i,query in enumerate(list(df["Question "])):
    print(f"question {i} = {query}")
    results1 = retriever.invoke(query)
    context=results1[0].page_content
    con.append(context)
    print(f"context {i} = {context}")
    prompt = f"""
    Use the context to answer the question, answer lies only in the context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    answer=llm.invoke(prompt).content
    print(f"answer {i} = {answer}")
    ans.append(answer)

In [ ]:
df["context"]=con

In [ ]:
df["answer"]=ans

In [ ]:
df.to_excel("QandA_BM25.xlsx")

RAG using MPNET embedding model

In [23]:
from langchain_community.embeddings import HuggingFaceEmbeddings


embedding_fn = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\DELL\AppData\Local\Temp\ipykernel_16584\2001995894.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_fn = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [24]:
df=pd.read_excel("Question and answer.xlsx")

In [25]:
vectordb2 = FAISS.load_local("mpnet", embedding_fn, allow_dangerous_deserialization=True)

In [26]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
con=[]
ans=[]
for i,query in enumerate(list(df["Question "])):
    print(f"question {i} = {query}")
    results1 = vectordb2.similarity_search(query, k=1)
    context=results1[0].page_content
    con.append(context)
    print(f"context {i} = {context}")
    prompt = f"""
    Use the context to answer the question, answer lies only in the context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    answer=llm.invoke(prompt).content
    print(f"answer {i} = {answer}")
    ans.append(answer)

question 0 = What does Article 150 of the Constitution of India provide regarding Government Accounts?
context 0 = provisions. The  Appropriation  Accounts  of the Union  Government  are submitted  to Parliament  
under  the provisions  of Article151  of the Constitution,  and are intended to  disclose - 
 
(a) That the moneys  indicated  therein  as having  been  disbursed,  were  legally  available  for and 
applicable to  the service  or purpose  to which they  had been applied or  charged;  
 
(b) That the expenditure conforms to the authority governing it , and
answer 0 = Article 150 of the Constitution of India provides that the Appropriation Accounts of the Union Government are to be submitted to Parliament.
question 1 = Where does the payment process in PFMS start?
context 1 = required checks sends it to bank with a payment advice in favour of beneficiaries. The payment is 
affected through the accredited bank to the beneficiaries and then a scroll is sent to PAO who 
takes it 

In [28]:
df["context"]=con

In [29]:
df["answer"]=ans

In [31]:
df.to_excel("QandA_MPNET.xlsx")

RAG using text-embedding-3-large

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

faiss_index1= FAISS.load_local(
    "faiss_store1",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
import pandas as pd
df=pd.read_excel("Question and answer.xlsx")

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
con=[]
ans=[]
for i,query in enumerate(list(df["Question "])):
    print(f"question {i} = {query}")
    results1 = faiss_index1.similarity_search(query, k=1)
    context=results1[0].page_content
    con.append(context)
    print(f"context {i} = {context}")
    prompt = f"""
    Use the context to answer the question, answer lies only in the context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    answer=llm.invoke(prompt).content
    print(f"answer {i} = {answer}")
    ans.append(answer)

In [ ]:
df["context"]=con

In [ ]:
df["answer"]=ans

In [ ]:
df.to_excel("QandA_embedding_3_large.xlsx")